In [2]:
%pip install \
    transformers \
    datasets \
    evaluate \
    rouge_score\
    loralib \
    bitsandbytes \
    peft --quiet

Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig, Trainer, DataCollatorForSeq2Seq
import evaluate
import pandas as pd

In [2]:
#copy toj=ken from https://huggingface.co/settings/tokens
import huggingface_hub
hf_token = 'hf_...'
# ابتدا login کنید
huggingface_hub.login(token=hf_token)

In [3]:
!huggingface-cli whoami

aapalireza


## Download Training Dataset

## 📘 Dataset Reference

**Title:** [Enhancing English-Persian Neural Machine Translation with a Large-Scale Parallel Dataset and Relative Position Representations](https://ieeexplore.ieee.org/abstract/document/10967409)  
**Authors:** Alireza Kamyab, Negar Baghaei Nejad, [Alireza Akhavanpour](https://class.vision/teacher/%D8%B9%D9%84%DB%8C%D8%B1%D8%B6%D8%A7-%D8%A7%D8%AE%D9%88%D8%A7%D9%86-%D9%BE%D9%88%D8%B1/)  
**Affiliation:** [Shenasa AI](https://shenasa.ai), Tehran, Iran  
**Dataset on HuggingFace:** [shenasa/English-Persian-Parallel-Dataset](https://huggingface.co/datasets/shenasa/English-Persian-Parallel-Dataset)


In [4]:
huggingface_dataset_name = "shenasa/English-Persian-Parallel-Dataset"
dataset = load_dataset(huggingface_dataset_name)

In [5]:
dataset = dataset['train']
dataset

Dataset({
    features: ['flash fire .', 'فلاش آتش .'],
    num_rows: 3960172
})

In [6]:
import pandas as pd
from datasets import Dataset


source_lang_col = 'src_lang'
target_lang_col = 'tgt_lang'

df_train = dataset.to_pandas()
df_train.columns = ['src_lang', 'tgt_lang']

dataset = Dataset.from_pandas(df_train)
dataset

Dataset({
    features: ['src_lang', 'tgt_lang'],
    num_rows: 3960172
})

### Keep only examples where the target translation is a valid string,
#### filtering out any records with missing or non-string translations.

In [7]:
dataset = dataset.filter(lambda x: isinstance(x[target_lang_col], str))
dataset = dataset.filter(lambda x: isinstance(x[source_lang_col], str))
dataset

Filter:   0%|          | 0/3960172 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3960159 [00:00<?, ? examples/s]

Dataset({
    features: ['src_lang', 'tgt_lang'],
    num_rows: 3960158
})

In [8]:
source_lang_col = dataset.column_names[0]
target_lang_col = dataset.column_names[1]
for i in range(0,100,20):
    print(f"Source: {dataset[i][source_lang_col]}")
    print(f"Target: {dataset[i][target_lang_col]}")
    print("-" * 50)

Source: superheats the air . burns the lungs like rice paper .
Target: هوا را فوق العاده گرم می کند . ریه ها را مثل کاغذ برنج می سوزاند .
--------------------------------------------------
Source: you gave elena a chance .
Target: تو به النا فرصت دادی
--------------------------------------------------
Source: chris . chris , come here .
Target: کریس . کریس بیا اینجا
--------------------------------------------------
Source: l cant find an access hatch . baby , look at me . look at me .
Target: من نمی توانم دریچه دسترسی پیدا کنم . عزیزم به من نگاه کن به من نگاه کن .
--------------------------------------------------
Source: look , baby , the waters gon na rise , and l am right here , okay .
Target: نگاه کن عزیزم ، آبها بالا می آیند ، و من اینجا هستم ، باشه .
--------------------------------------------------


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig

model_name = "google/gemma-2-9b-it"

bnbConfig = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "auto",
    quantization_config=bnbConfig
)

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

... (3 outputs left out)

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [10]:
def print_number_of_trainable_model_parameters(model):
    all_model_params = model.num_parameters()
    trainable_model_params = sum(param.numel() for param in model.parameters() if param.requires_grad)

    percentage_trainable = 100 * trainable_model_params / all_model_params if all_model_params > 0 else 0

    return (f"Trainable model parameters: {trainable_model_params}\n"
            f"All model parameters: {all_model_params}\n"
            f"Percentage of trainable model parameters: {percentage_trainable:.2f}%")

print(print_number_of_trainable_model_parameters(model))

Trainable model parameters: 918109696
All model parameters: 9241705984
Percentage of trainable model parameters: 9.93%


In [11]:
index = 200

source_lang = dataset[index][source_lang_col]
target_lang = dataset[index][target_lang_col]

prompt = f"""
Translate the following English text to Persian:
English: {source_lang}

Persian translation:
"""
print(prompt)



Translate the following English text to Persian:
English: direct upwards first door to the left .

Persian translation:



In [12]:
device = torch.device("cuda")
inputs = tokenizer(prompt, return_tensors='pt').to(device)

In [13]:
inputs

{'input_ids': tensor([[     2,    108,  49688,    573,   2412,   4645,   2793,    577,  48848,
         235292,    108,  12023, 235292,   3280,  45509,   1370,   4444,    577,
            573,   2731,    954,    109, 146233,  17183, 235292,    108]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]], device='cuda:0')}

In [14]:
model = model.eval()
with torch.no_grad():
  output = tokenizer.decode(
      model.generate(
          inputs["input_ids"],
          max_new_tokens=100,
      )[0],
      skip_special_tokens=True
  )
print(output)


Translate the following English text to Persian:
English: direct upwards first door to the left .

Persian translation:
در ورودی اول را به سمت چپ و سپس به سمت بالا مستقیم بروید.






In [15]:
index = 95000

source_lang = dataset[index][source_lang_col]
target_lang = dataset[index][target_lang_col]

prompt = f"""
Translate the following English text to Persian:
English: {source_lang}

Persian translation:
"""

device = torch.device("cuda")
inputs = tokenizer(prompt, return_tensors='pt').to(device)
with torch.no_grad():
  output = tokenizer.decode(
      model.generate(
          inputs["input_ids"],
          max_new_tokens=100,
          do_sample=False
      )[0],
      skip_special_tokens=True
  )

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'BASELINE Translation:\n{target_lang}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Translate the following English text to Persian:
English: Sweat had broken on his forehead like dew.

Persian translation:

---------------------------------------------------------------------------------------------------
BASELINE Translation:
عرق مثل شبنم روی پیشانی اش نشسته بود .

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:

Translate the following English text to Persian:
English: Sweat had broken on his forehead like dew.

Persian translation:
عرق بر پیشانی اش مانند قطرات شبنم ریخته بود.




